In [ ]:
import duckdb
from networks.cnn_dilated_final_version import CNNModel
from networks.cnn_deep import build_dataloaders

con = duckdb.connect('../capillary.db')
negatives = con.execute(""" 
                 SELECT row_id FROM protein_data WHERE array_contains(analysis,'0085') AND interpretation ILIKE '%ingen%m%komponent%påvisas%immunfixation%utförd%';
                 """).fetchnumpy()['row_id']
positives = con.execute(""" 
                 SELECT row_id FROM protein_data WHERE array_contains(analysis,'0085') AND interpretation ILIKE '%nyupptäckt%m%komponent%';
                 """).fetchnumpy()['row_id']

df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 """).df()
con.close()
negatives = set(negatives)
positives = set(positives)

df = df[df['row_id'].isin(negatives | positives)]
df['label'] = df['row_id'].apply(
    lambda x: 1 if x in positives else 0
)


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 1].sample(frac=0.7).index


train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")
CNN = CNNModel()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
CNN.reset_weights()
CNN.retrain(cnn_train_dl,cnn_val_dl,patience=15)

con = duckdb.connect('../capillary.db')
negatives = con.execute(""" 
                 SELECT row_id FROM protein_data WHERE array_contains(analysis,'0085') AND interpretation ILIKE '%ingen%m%komponent%påvisas%immunfixation%utförd%';
                 """).fetchnumpy()['row_id']
positives = con.execute(""" 
                 SELECT row_id FROM protein_data WHERE array_contains(analysis,'0085') AND interpretation ILIKE '%nyupptäckt%m%komponent%';
                 """).fetchnumpy()['row_id']

df = con.execute(""" 
                 SELECT row_id, value,fractions, boundaries,albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm, label, set, interpretation
                 FROM protein_data
                 """).df()
con.close()
negatives = set(negatives)
positives = set(positives)

df = df[df['row_id'].isin(negatives | positives)]
df['label'] = df['row_id'].apply(
    lambda x: 1 if x in positives else 0
)


train_rows = df[df['set'] == 'train']
val_rows   = df[df['set'] == 'val']
test_rows  = df[df['set'] == 'test']

k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


CNN = CNNModel()
CNN.retrain_with_k_fold(k_fold_rows)

Antal utan m-komponent i träningsdatan: 6317
Antal med m-komponent i träningsdatan: 902
Total parameters: 324,898
  -> ny bästa modell sparad till ../models/cnn_final_dilation.pth
Epoch   0 | train: 0.7035 | val: 0.6712 | acc: 32.11% | AUC: 0.736  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_final_dilation.pth
Epoch   1 | train: 0.5309 | val: 0.5388 | acc: 42.75% | AUC: 0.806  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_final_dilation.pth
Epoch   2 | train: 0.3969 | val: 0.5850 | acc: 86.07% | AUC: 0.901  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_final_dilation.pth
Epoch   3 | train: 0.3781 | val: 0.2375 | acc: 83.95% | AUC: 0.923  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_final_dilation.pth
Epoch   4 | train: 0.3277 | val: 0.3805 | acc: 89.36% | AUC: 0.934  | LR: 0.001
  -> ny bästa modell sparad till ../models/cnn_final_dilation.pth
Epoch   5 | train: 0.2976 | val: 0.3554 | acc: 89.94% | AUC: 0.951  | LR: 0.001
Epoch   6 

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.161547,0.246058,71.042471,0.943899,0.270000,0.987421,54,146,4,314
1,2,0.127702,0.070646,87.040619,0.967305,0.703518,0.974843,140,59,8,310
2,3,0.150744,0.263175,61.389961,0.942421,0.000000,1.000000,0,200,0,318
3,4,0.144366,0.104282,78.185328,0.936274,0.490000,0.965409,98,102,11,307
4,5,0.137380,0.224108,79.690522,0.959338,0.505000,0.981073,101,99,6,311
5,6,0.135696,0.224114,61.315280,0.971128,0.000000,1.000000,0,200,0,317
6,7,0.125823,0.197731,82.011605,0.933060,0.605000,0.955836,121,79,14,303
7,8,0.122273,0.103537,87.209302,0.957215,0.768844,0.936909,153,46,20,297
8,9,0.135686,0.282272,81.044487,0.962397,0.530000,0.987382,106,94,4,313
9,10,0.072131,0.120657,88.781431,0.954890,0.855000,0.908517,171,29,29,288


In [17]:
predict_fn = get_predict_fn(cnn_suffix = 'immunofixation',ae_suffix='first_time')
test_rows = test_rows[test_rows['label'].isin([0,1])]

result = predict_fn(test_rows)
evaluate(result,threshold=0.5,proportion=70)

              precision    recall  f1-score   support

     Negativ       0.93      0.78      0.85       745
     Positiv       0.66      0.88      0.75       351

    accuracy                           0.81      1096
   macro avg       0.80      0.83      0.80      1096
weighted avg       0.84      0.81      0.82      1096

[[584 161]
 [ 42 309]]
Accuracy:  81.48%
FN-rate:   11.97%  (farliga missade fall)
FP-rate:   21.61%  (onödiga larm)


(array([0.9937115 , 0.7772805 , 0.9694337 , ..., 0.939833  , 0.18907686,
        0.25031412], shape=(1096,), dtype=float32),
 array([1, 1, 1, ..., 1, 0, 0], shape=(1096,)),
 20        1
 108       1
 505       1
 790       1
 798       1
          ..
 172182    0
 172197    1
 172201    1
 172211    0
 172378    0
 Name: proportion_gamma_region, Length: 1096, dtype: int64,
 array([155130, 153295, 144120, ..., 176094, 177036, 148187], shape=(1096,)))